# 1. Подготовка и методология

## 1.1. Методология расчета циклов восстановления (Recovery Time)

Для объективного анализа устойчивости валют мы используем концепцию **High-Water Mark (HWM)** и делим историю движения курса на завершенные и незавершенные циклы.

### Ключевые определения:
1. **High-Water Mark (HWM):** Исторический максимум абсолютного курса на временном отрезке от начала наблюдений до текущей даты.
2. **Состояние просадки (Under Water):** Период, когда текущий курс находится ниже зафиксированного ранее HWM.
3. **Цикл восстановления (Recovery Cycle):** Интервал времени, который включает три фазы:
    * **Пик (Peak):** Точка достижения нового HWM.
    * **Дно (Trough):** Точка максимального падения (MDD) внутри цикла.
    * **Восстановление (Recovery):** Точка первого возврата курса к уровню предыдущего HWM.

### Математические параметры:
* **Время падения (Drawdown Duration):** Количество календарных дней от *Пика* до *Дна*.
* **Время восстановления (Recovery Time):** Количество календарных дней от *Дна* до точки *Восстановления*.
* **Полный цикл:** Общее время нахождения «под водой» (от Пика до Восстановления).

#### Правила фильтрации и обработки данных:
* **Порог значимости:** Чтобы исключить технический шум, в анализе учитываются только циклы с просадкой (MDD) более **0.5%**.
* **Открытые просадки:** Если на текущую дату курс не вернулся к HWM, цикл считается «открытым». Для таких случаев рассчитывается текущее время нахождения в просадке, но они не включаются в расчет *среднего* времени восстановления, чтобы не занижать исторические показатели.
* **Абсолютные координаты:** Все расчеты производятся на базе абсолютных курсов [abscur.ru](https://www.abscur.ru), что исключает влияние циклов отдельных валют-якорей (например, USD).

## 1.2. Специфика абсолютных курсов: Очистка от «долларового шума»

Традиционный анализ валютных рисков через пары (например, USD/TRY или USD/EGP) страдает от фундаментального изъяна: **индикатор всегда зависит от состояния второй валюты (якоря).**

Если доллар США (USD) начинает цикл глобального укрепления из-за роста ставок ФРС, на графиках парных курсов кажется, что «падают» абсолютно все валюты мира. Это создает ложные сигналы о системном кризисе там, где его нет.

### В чем преимущество анализа просадок в абсолютных курсах:

1. **Изоляция локального риска:** Абсолютный курс рассчитывается относительно широкой корзины мировых валют. Если валюта страны X падает в абсолютном выражении, это означает реальный отток капитала из этой экономики, а не просто временную силу доллара.
2. **Истинное время восстановления:** В валютной паре актив может «восстановиться» только потому, что ослаб якорь (доллар), при этом реальная покупательная способность национальной валюты останется на дне. Абсолютный курс фиксирует восстановление только тогда, когда валюта реально возвращает свои позиции относительно всей мировой системы.
3. **Объективность циклов:** Использование данных [abscur.ru](https://www.abscur.ru) позволяет отделить глобальные макроэкономические циклы (связанные с эмиссией резервных валют) от индивидуальных циклов устойчивости каждой конкретной страны.

**Вывод:** Анализ времени восстановления в абсолютных координатах дает «чистую» картину жизнеспособности финансовой системы страны, исключая влияние внешних спекулятивных факторов.

# 2. Разработка алгоритма (Python)

## Загрузка актуального датасета `abscur.csv`.

In [1]:
import pandas as pd

# Путь к файлу из подключенного датасета
file_path = '/kaggle/input/notebooks/eavprog/abscur2/abscur.csv'

# Загрузка данных с распознаванием дат
df = pd.read_csv(file_path, parse_dates=['Date'])

# Сортируем по дате (на случай, если в CSV порядок нарушен)
df = df.sort_values('Date')

# Устанавливаем дату как индекс — так удобнее делать срезы по времени (DateOffset)
df.set_index('Date', inplace=True)

# Проверка: выводим информацию о датасете и первые строки
print(f"Загружено строк: {len(df)}")
print(f"Диапазон дат: {df.index.min().date()} — {df.index.max().date()}")
print(f"Количество валют: {len(df.columns)}")

df.head()

Загружено строк: 5898
Диапазон дат: 2006-07-28 — 2026-03-16
Количество валют: 45


,AED,ARS,AUD,BRL,CAD,CHF,CLP,CNY,COP,CZK,...,SAR,SEK,SGD,THB,TRY,TWD,UAH,USD,VND,ZAR
Date,,,,,,,,,,,,,,,,,,,,,
2006-07-28,3.898841,1.683174,10.838947,5.596110,12.292434,15.511844,0.024427,1.827150,0.006823,0.649023,...,3.817157,1.655008,11.441231,0.436865,6.317193,0.443440,0.989630,14.320445,0.000673,1.282037
2006-07-31,3.898831,1.683169,10.838955,5.596095,12.292443,15.511855,0.024427,1.827266,0.006822,0.649024,...,3.817146,1.655010,11.441239,0.436864,6.317176,0.443439,0.989628,14.320406,0.000673,1.282038
2006-08-01,3.898863,1.683183,10.838931,5.596140,12.292415,15.511820,0.024427,1.826917,0.006823,0.649022,...,3.817177,1.655006,11.441214,0.436868,6.317227,0.443442,0.989636,14.320523,0.000673,1.282035
2006-08-02,3.898834,1.683171,10.838953,5.596099,12.292440,15.511852,0.024427,1.827231,0.006822,0.649024,...,3.817149,1.655009,11.441237,0.436864,6.317181,0.443439,0.989628,14.320418,0.000673,1.282038
2006-08-03,3.898884,1.683192,10.838915,5.596171,12.292397,15.511797,0.024427,1.826685,0.006823,0.649021,...,3.817198,1.655003,11.441197,0.436870,6.317262,0.443445,0.989641,14.320600,0.000673,1.282032
